In [32]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_data.csv")

df.shape
df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,mort_acc,pub_rec_bankruptcies,default
0,32000.0,60 months,10.49,687.65,B,B3,10+ years,MORTGAGE,120000.0,Verified,...,0.0,20.0,0.0,39687.0,57.8,42.0,Individual,2.0,0.0,0
1,9600.0,36 months,12.99,323.42,C,C1,NaN,RENT,21900.0,Verified,...,1.0,13.0,1.0,4509.0,38.9,20.0,Individual,0.0,1.0,0
2,4000.0,36 months,6.68,122.93,A,A3,4 years,MORTGAGE,83000.0,Not Verified,...,0.0,16.0,0.0,1564.0,17.2,25.0,Individual,2.0,0.0,0
3,6025.0,36 months,10.91,197.00,B,B4,10+ years,RENT,52000.0,Not Verified,...,1.0,11.0,0.0,2706.0,12.8,25.0,Individual,0.0,0.0,0
4,25000.0,60 months,26.30,752.96,E,E5,10+ years,OWN,65000.0,Verified,...,1.0,19.0,0.0,49461.0,24.7,33.0,Individual,0.0,0.0,0


In [33]:
# fico average
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

# income to loan
df["income_to_loan"] = df["annual_inc"] / df["loan_amnt"]

# installment burden
df["installment_to_income"] = df["installment"] / (df["annual_inc"] / 12)

# clean revol_util
df["revol_util"] = pd.to_numeric(
    df["revol_util"].astype(str).str.replace("%", "", regex=False),
    errors="coerce"
)

# term to numeric
df["term"] = (
    df["term"]
    .astype(str)
    .str.replace(" months", "", regex=False)
    .str.strip()
)
df["term"] = pd.to_numeric(df["term"], errors="coerce")

# emp_length to numeric
df["emp_length"] = (
    df["emp_length"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)
df["emp_length"] = pd.to_numeric(df["emp_length"], errors="coerce")

df = df.drop(columns=["fico_range_low", "fico_range_high"], errors="ignore")

In [34]:
df.isnull().mean().sort_values(ascending=False).head(10)

emp_length              0.06458
mort_acc                0.02122
dti                     0.00084
revol_util              0.00066
pub_rec_bankruptcies    0.00052
installment             0.00004
term                    0.00004
int_rate                0.00004
grade                   0.00004
loan_amnt               0.00004
dtype: float64

In [39]:
# Fill numeric
df[df.select_dtypes(include="number").columns] = df.select_dtypes(include="number").fillna(
    df.median(numeric_only=True)
)

# Split
y = df["default"]
X = df.drop(columns=["default"])

# Encode
X = pd.get_dummies(X, drop_first=True)

# FINAL SAFETY
X = X.fillna(0)

# Check
print(X.isnull().sum().sum())  # should be 0

0


In [40]:
df.shape
df.isnull().sum().sum()

np.int64(24)

In [41]:
X.to_csv("../data/processed/X.csv", index=False)
y.to_csv("../data/processed/y.csv", index=False)